<a href="https://colab.research.google.com/github/nghialv108/vietnamese-foods-classification/blob/main/01_eda_30vnfoods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("quandang/vietnamese-foods")

print("Path to dataset files:", path)

from pathlib import Path

print(Path(path).iterdir(),)

100%|██████████| 4.17G/4.17G [00:47<00:00, 93.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/quandang/vietnamese-foods/versions/11


In [8]:
from pathlib import Path

dataset_root = Path("/root/.cache/kagglehub/datasets/quandang/vietnamese-foods/versions/11")

for item in sorted(dataset_root.iterdir()):
    print(item.name, "| thư mục" if item.is_dir() else "| file")

inner = dataset_root / "Images"

for item in sorted(inner.iterdir()):
    print(item.name, "| thư mục" if item.is_dir() else "| file")

Images | thư mục
LICENSE | file
Urls | thư mục
Test | thư mục
Train | thư mục
Validate | thư mục


In [9]:
from pathlib import Path

dataset_root = Path("/root/.cache/kagglehub/datasets/quandang/vietnamese-foods/versions/11")

def show_tree(folder, depth=0, max_depth=2, max_items=6):
    items = sorted(folder.iterdir())
    for item in items[:max_items]:
        print("  " * depth + item.name + ("/" if item.is_dir() else ""))
        if item.is_dir() and depth < max_depth:
            show_tree(item, depth + 1, max_depth, max_items)
    if len(items) > max_items:
        print("  " * depth + f"... (còn {len(items) - max_items} mục nữa)")

show_tree(dataset_root)

Images/
  Test/
    Banh beo/
    Banh bot loc/
    Banh can/
    Banh canh/
    Banh chung/
    Banh cuon/
    ... (còn 24 mục nữa)
  Train/
    Banh beo/
    Banh bot loc/
    Banh can/
    Banh canh/
    Banh chung/
    Banh cuon/
    ... (còn 24 mục nữa)
  Validate/
    Banh beo/
    Banh bot loc/
    Banh can/
    Banh canh/
    Banh chung/
    Banh cuon/
    ... (còn 24 mục nữa)
LICENSE
Urls/
  Banh beo.txt
  Banh bot loc.txt
  Banh can.txt
  Banh canh.txt
  Banh chung.txt
  Banh cuon.txt
  ... (còn 24 mục nữa)


In [10]:
import pandas as pd
from pathlib import Path

dataset_root = Path("/root/.cache/kagglehub/datasets/quandang/vietnamese-foods/versions/11")

# Tìm thư mục chứa train/test/validation, kể cả khi nằm lồng bên trong
base = next(p.parent for p in dataset_root.rglob("*") if p.is_dir() and p.name.lower() == "train")
print("Thư mục gốc chứa các tập:", base)
print("Các mục ở đó:", sorted(p.name for p in base.iterdir()))

Thư mục gốc chứa các tập: /root/.cache/kagglehub/datasets/quandang/vietnamese-foods/versions/11/Images
Các mục ở đó: ['Test', 'Train', 'Validate']


In [11]:
splits = {
    "train": base / "Train",
    "val":   base / "Validate",   # sửa nếu tên khác, ví dụ "val"
    "test":  base / "Test",
}

rows = []
for split_name, split_dir in splits.items():
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        n = sum(1 for f in class_dir.iterdir() if f.is_file())
        rows.append({"split": split_name, "class": class_dir.name, "count": n})

df = pd.DataFrame(rows)
table = df.pivot(index="class", columns="split", values="count")
table["total"] = table.sum(axis=1)
print(table)
print()
print(table.sum())

split             test  train  val  total
class                                    
Banh beo           129    451   65    645
Banh bot loc       144    503   73    720
Banh can           149    520   75    744
Banh canh          193    674   97    964
Banh chung         102    354   50    506
Banh cuon          228    798  114   1140
Banh duc           133    463   66    662
Banh gio           129    448   64    641
Banh khot          167    584   84    835
Banh mi            268    935  133   1336
Banh pia            89    310   45    444
Banh tet           138    480   68    686
Banh trang nuong   159    556   80    795
Banh xeo           235    821  117   1173
Bun bo Hue         306   1071  153   1530
Bun dau mam tom    184    640   92    916
Bun mam            155    541   77    773
Bun rieu           231    805  115   1151
Bun thit nuong     150    522   75    747
Ca kho to          136    473   67    676
Canh chua          165    577   83    825
Cao lau            124    432   62

In [12]:
names = {k: set(p.name for p in v.iterdir() if p.is_dir()) for k, v in splits.items()}
print("Train = Val:", names["train"] == names["val"])
print("Train = Test:", names["train"] == names["test"])
print("Số lớp:", len(names["train"]))

Train = Val: True
Train = Test: True
Số lớp: 30
